In [ ]:
import sys
import random
import time

def solve():
    # Ghi nhận thời gian để giới hạn thời gian chạy (tránh TLE)
    start_time = time.time()
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    N = int(input_data[0])
    K = int(input_data[1])

    # Đọc mảng thời gian bảo trì (chèn 0 ở vị trí số 0 là kho)
    d = [0] * (N + 1)
    idx = 2
    for i in range(1, N + 1):
        d[i] = int(input_data[idx])
        idx += 1

    # Đọc ma trận thời gian di chuyển
    t = []
    for i in range(N + 1):
        row = []
        for j in range(N + 1):
            row.append(int(input_data[idx]))
            idx += 1
        t.append(row)

    best_routes = None
    best_max_cost = float('inf')

    # Seed cố định để reproducible nếu cần (hoặc có thể bỏ)
    random.seed(42)

    # Chạy vòng lặp trong giới hạn thời gian ~1.5 giây
    while time.time() - start_time < 1.5:
        # 1. BƯỚC KHỞI TẠO (GREEDY NGUYÊN TẮC KẾT HỢP RANDOM)
        routes = [[0] for _ in range(K)]
        costs = [0] * K
        unvisited = list(range(1, N + 1))

        while unvisited:
            # Chọn nhân viên đang có tổng thời gian ít nhất
            k = min(range(K), key=lambda x: costs[x])
            last = routes[k][-1]

            # Đánh giá khách hàng ưu tiên (gần nhất)
            candidates = []
            for v in unvisited:
                inc = t[last][v] + d[v]
                candidates.append((inc, v))

            candidates.sort(key=lambda x: x[0])
            # Chọn ngẫu nhiên trong top 3 khách hàng tốt nhất để tăng tính đa dạng
            limit = min(len(candidates), 3)
            chosen_inc, chosen_v = random.choice(candidates[:limit])

            routes[k].append(chosen_v)
            costs[k] += chosen_inc
            unvisited.remove(chosen_v)

        # Trở về điểm xuất phát (địa điểm 0)
        for k in range(K):
            costs[k] += t[routes[k][-1]][0]
            routes[k].append(0)

        # 2. BƯỚC TÌM KIẾM CỤC BỘ (LOCAL SEARCH)
        improved = True
        while improved:
            improved = False
            # Tập trung tối ưu tuyến đang chịu chi phí lớn nhất
            max_r = max(range(K), key=lambda x: costs[x])

            # Phép toán 1: Relocate (Chuyển 1 điểm từ max_r sang route khác)
            for i in range(1, len(routes[max_r]) - 1):
                v = routes[max_r][i]
                rem_delta = -t[routes[max_r][i-1]][v] - d[v] - t[v][routes[max_r][i+1]] + t[routes[max_r][i-1]][routes[max_r][i+1]]
                new_max_cost = costs[max_r] + rem_delta

                for r in range(K):
                    if r == max_r: continue
                    best_add_delta = float('inf')
                    best_j = -1
                    for j in range(1, len(routes[r])):
                        add_delta = -t[routes[r][j-1]][routes[r][j]] + t[routes[r][j-1]][v] + d[v] + t[v][routes[r][j]]
                        if add_delta < best_add_delta:
                            best_add_delta = add_delta
                            best_j = j

                    # Nếu thời gian mới của max_r VÀ r đều nhỏ hơn max hiện tại -> áp dụng
                    if new_max_cost < costs[max_r] and (costs[r] + best_add_delta) < costs[max_r]:
                        routes[r].insert(best_j, v)
                        routes[max_r].pop(i)
                        costs[r] += best_add_delta
                        costs[max_r] = new_max_cost
                        improved = True
                        break
                if improved: break
            if improved: continue

            # Phép toán 2: Swap (Hoán đổi 1 điểm giữa max_r và route khác)
            for i in range(1, len(routes[max_r]) - 1):
                v = routes[max_r][i]
                for r in range(K):
                    if r == max_r: continue
                    for j in range(1, len(routes[r]) - 1):
                        u = routes[r][j]
                        delta_max = -t[routes[max_r][i-1]][v] - d[v] - t[v][routes[max_r][i+1]] \
                                    + t[routes[max_r][i-1]][u] + d[u] + t[u][routes[max_r][i+1]]

                        delta_r = -t[routes[r][j-1]][u] - d[u] - t[u][routes[r][j+1]] \
                                  + t[routes[r][j-1]][v] + d[v] + t[v][routes[r][j+1]]

                        if costs[max_r] + delta_max < costs[max_r] and costs[r] + delta_r < costs[max_r]:
                            routes[max_r][i] = u
                            routes[r][j] = v
                            costs[max_r] += delta_max
                            costs[r] += delta_r
                            improved = True
                            break
                    if improved: break
                if improved: break
            if improved: continue

            # Phép toán 3: 2-opt intra-route (Tối ưu thứ tự trong chính nhánh max_r)
            route = routes[max_r]
            for i in range(1, len(route) - 2):
                for j in range(i + 1, len(route) - 1):
                    old_seg = 0
                    for idx in range(i-1, j+1):
                        old_seg += t[route[idx]][route[idx+1]]

                    new_seg = t[route[i-1]][route[j]] + t[route[i]][route[j+1]]
                    for idx in range(j, i, -1):
                        new_seg += t[route[idx]][route[idx-1]]

                    delta = new_seg - old_seg
                    if costs[max_r] + delta < costs[max_r]:
                        route[i:j+1] = route[i:j+1][::-1]
                        costs[max_r] += delta
                        improved = True
                        break
                if improved: break

        # Ghi nhận kết quả tối ưu nhất trong số các vòng lặp ngẫu nhiên
        curr_max = max(costs)
        if curr_max < best_max_cost:
            best_max_cost = curr_max
            best_routes = [r[:] for r in routes]

    # 3. IN KẾT QUẢ ĐẦU RA (THEO FORMAT)
    print(K)
    for k in range(K):
        print(len(best_routes[k]))
        print(" ".join(map(str, best_routes[k])))

if __name__ == '__main__':
    solve()
